In [1]:
import os
import json
import glob
import re
import matplotlib.pyplot as plt
import pandas as pd

def plot_knn_purity(folder_path, output_file=None):
    """
    Reads knn_results_cp-*.json files from the folder, extracts purity scores,
    and plots a line chart of scores vs steps.
    Highlights the maximum score for each k value.
    """
    # 确定输出文件路径
    if output_file is None:
        target_output = os.path.join(folder_path, "purity_scores_plot.png")
    else:
        target_output = output_file

    # 检查输出文件是否已存在
    if os.path.exists(target_output):
        print(f"Skipping {folder_path}: Plot already exists at {target_output}")
        return

    # 获取所有符合模式的 JSON 文件
    json_files = glob.glob(os.path.join(folder_path, "knn_results_cp-*.json"))
    
    data = []
    
    # 如果没有找到文件，也打印一下信息（可选）
    if not json_files:
        print(f"No json files found in {folder_path}")
        return
        
    print(f"Processing {folder_path}: Found {len(json_files)} files")
    
    for file_path in json_files:
        filename = os.path.basename(file_path)
        # 从文件名提取 step: "knn_results_cp-{step}.json"
        match = re.search(r"knn_results_cp-(\d+)\.json", filename)
        if match:
            step = int(match.group(1))
            
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = json.load(f)
                    purity_scores = content.get("purity_scores", {})
                    
                    # 提取 k=10 和 k=30 的分数
                    score_10 = purity_scores.get("10")
                    score_30 = purity_scores.get("30")
                    
                    if score_10 is not None and score_30 is not None:
                        data.append({
                            "step": step,
                            "k=10": score_10,
                            "k=30": score_30
                        })
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    
    if not data:
        print(f"No valid data extracted in {folder_path}.")
        return

    # 创建 DataFrame 并按 step 排序
    df = pd.DataFrame(data)
    df = df.sort_values("step")
    
    # 开始绘图
    plt.figure(figsize=(12, 6))
    
    # --- 绘制 k=10 ---
    plt.plot(df["step"], df["k=10"], marker='o', label='k=10', linewidth=2, alpha=0.7)
    # 标注所有点
    for x, y in zip(df["step"], df["k=10"]):
        plt.text(x, y, f"{y:.4f}", ha='center', va='bottom', fontsize=8, alpha=0.7)
        
    # 寻找并高亮 k=10 的最高点
    if not df.empty:
        max_k10 = df.loc[df['k=10'].idxmax()]
        plt.plot(max_k10['step'], max_k10['k=10'], 'r*', markersize=15, zorder=10) # 红色五角星
        plt.annotate(f"Max: {max_k10['k=10']:.4f}\n(Step {int(max_k10['step'])})", 
                     xy=(max_k10['step'], max_k10['k=10']), 
                     xytext=(0, 30), textcoords='offset points', 
                     ha='center', color='darkred', fontweight='bold',
                     arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=.2", color='darkred'))

    # --- 绘制 k=30 ---
    plt.plot(df["step"], df["k=30"], marker='s', label='k=30', linewidth=2, alpha=0.7)
    # 标注所有点
    for x, y in zip(df["step"], df["k=30"]):
        plt.text(x, y, f"{y:.4f}", ha='center', va='top', fontsize=8, alpha=0.7)

    # 寻找并高亮 k=30 的最高点
    if not df.empty:
        max_k30 = df.loc[df['k=30'].idxmax()]
        plt.plot(max_k30['step'], max_k30['k=30'], 'r*', markersize=15, zorder=10) # 红色五角星
        plt.annotate(f"Max: {max_k30['k=30']:.4f}\n(Step {int(max_k30['step'])})", 
                     xy=(max_k30['step'], max_k30['k=30']), 
                     xytext=(0, -40), textcoords='offset points', 
                     ha='center', color='darkred', fontweight='bold',
                     arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=-.2", color='darkred'))
    
    plt.title(f"kNN Neighborhood Purity vs Step\n({os.path.basename(folder_path)})") # 标题增加文件夹名区分
    plt.xlabel("Step")
    plt.ylabel("Purity Score")
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    
    plt.savefig(target_output)
    print(f"Plot saved to {target_output}")
    plt.close()

In [3]:
import glob
import os

root_dir = "/storage/BioMedNLP/llm2vec/visualization/knn_results"
# 获取三级路径
paths = glob.glob(os.path.join(root_dir, "*", "*", "*"))
# 过滤只保留文件夹
subfolders = [p for p in paths if os.path.isdir(p)]

# for p in subfolders:
#     print(p)
for folder in subfolders:
    plot_knn_purity(folder)


Skipping /storage/BioMedNLP/llm2vec/visualization/knn_results/coarse/Derml2v-8B_MixCSE_ResAttn_DataV2/woinst: Plot already exists at /storage/BioMedNLP/llm2vec/visualization/knn_results/coarse/Derml2v-8B_MixCSE_ResAttn_DataV2/woinst/purity_scores_plot.png
Skipping /storage/BioMedNLP/llm2vec/visualization/knn_results/coarse/Derml2v-8B_MixCSE_ResAttn_DataV2/instV3: Plot already exists at /storage/BioMedNLP/llm2vec/visualization/knn_results/coarse/Derml2v-8B_MixCSE_ResAttn_DataV2/instV3/purity_scores_plot.png
Skipping /storage/BioMedNLP/llm2vec/visualization/knn_results/coarse/Derml2v-8B_MixCSE_DataV1/woinst: Plot already exists at /storage/BioMedNLP/llm2vec/visualization/knn_results/coarse/Derml2v-8B_MixCSE_DataV1/woinst/purity_scores_plot.png
Skipping /storage/BioMedNLP/llm2vec/visualization/knn_results/coarse/Derml2v-8B_MixCSE_DataV1/instV3: Plot already exists at /storage/BioMedNLP/llm2vec/visualization/knn_results/coarse/Derml2v-8B_MixCSE_DataV1/instV3/purity_scores_plot.png
Processi